In [0]:
SELECT 
    COUNT(order_id) AS total_orders 
FROM alan_orders_bronze;

total_orders
16651


In [0]:
SELECT 
    payment_method, 
    COUNT(order_id) AS total_orders
FROM alan_orders_bronze
GROUP BY payment_method
ORDER BY total_orders DESC;

payment_method,total_orders
cash,4201
card,4186
wallet,4140
UPI,4124


In [0]:
SELECT 
    order_status, 
    COUNT(order_id) AS total_orders
FROM alan_orders_bronze
GROUP BY order_status
ORDER BY total_orders DESC;

order_status,total_orders
en route,4238
preparing,4179
cancelled,4129
delivered,4105


In [0]:
SELECT 
    restaurant_id, 
    COUNT(order_id) AS total_orders
FROM alan_orders_bronze
GROUP BY restaurant_id
ORDER BY total_orders DESC;

restaurant_id,total_orders
R005,114
R198,111
R031,110
R131,108
R083,108
R171,108
R058,107
R092,106
R035,104
R138,104


In [0]:
SELECT 
    customer_id, 
    COUNT(order_id) AS total_orders
FROM alan_orders_bronze
GROUP BY customer_id
ORDER BY total_orders DESC;

customer_id,total_orders
C7381,9
C6444,8
C0362,8
C2231,8
C6672,8
C1449,8
C5365,7
C1423,7
C9140,7
C9870,7


In [0]:
SELECT 
    agent_id, 
    COUNT(order_id) AS total_orders
FROM alan_orders_bronze
GROUP BY agent_id
ORDER BY total_orders DESC;

agent_id,total_orders
A309,117
A207,114
A393,99
A350,91
A138,88
A245,86
A168,86
A315,84
A370,83
A056,81


In [0]:
SELECT 
    order_id, 
    customer_id,
    restaurant_id,
    total_amount, 
    payment_method, 
    order_status
FROM alan_orders_bronze
ORDER BY total_amount DESC
LIMIT 10;

order_id,customer_id,restaurant_id,total_amount,payment_method,order_status
c12860c3-683a-47f7-b790-7cbbbcfc0fec,C5726,R159,361.46,cash,en route
05e56a7c-190c-4bde-ac13-e34b38fe155a,C5959,R151,355.9,cash,delivered
341cb220-46e4-4f0d-b2f2-2e494b5729dd,C6763,R147,352.45,card,en route
3b6f5bd2-08e4-412d-be9e-7701463015b0,C3608,R105,348.95,card,preparing
f43228bd-94e5-4ef6-914a-c7ebdceaa572,C4740,R056,342.91,wallet,delivered
621d746e-bd7d-4e27-b06e-a098a9bffda7,C0287,R151,340.65,cash,delivered
75dfeba8-60e0-42c2-9149-315f030e8d9d,C6393,R050,333.85,card,cancelled
72559328-7d0d-4177-8c44-4ebc43dca731,C3804,R171,332.98,wallet,preparing
b9dc92fc-1922-4274-a42c-85b83e46bea2,C0791,R091,327.8,wallet,delivered
7d9b6852-ffb2-4bac-84e4-cc7960284e30,C5690,R122,327.44,card,preparing


In [0]:
SELECT 
    payment_method, 
    CEIL(AVG(tip) * 100) / 100 AS avg_tip_amount
FROM alan_orders_bronze
GROUP BY payment_method;

payment_method,avg_tip_amount
wallet,2.5
cash,2.48
card,2.51
UPI,2.53


In [0]:
SELECT
    CAST(CEIL((COUNT(
        CASE
            WHEN tip > 0.10 * total_amount THEN 1
        END
    ) * 100.0 / COUNT(*)
    ) * 100
    ) / 100
    AS DECIMAL(10,2)
    ) AS percentage_high_tip
FROM alan_orders_bronze;

percentage_high_tip
21.49


In [0]:
SELECT
    order_status,
    CAST(CEIL(
        COUNT(*) * 100.0 / 
        (SELECT COUNT(*) 
        FROM alan_orders_bronze) 
        * 100
        ) / 100
        AS DECIMAL(10,2)
    ) AS percentage
FROM alan_orders_bronze
GROUP BY order_status
ORDER BY order_status;

order_status,percentage
cancelled,24.80
delivered,24.66
en route,25.46
preparing,25.10


In [0]:
SELECT 
    restaurant_id, 
    COUNT(order_id) AS total_orders 
FROM alan_orders_bronze 
GROUP BY restaurant_id 
ORDER BY total_orders DESC 
LIMIT 5;

restaurant_id,total_orders
R005,114
R198,111
R031,110
R171,108
R083,108


In [0]:
SELECT 
    customer_id, 
    COUNT(order_id) AS total_orders 
FROM alan_orders_bronze 
GROUP BY customer_id 
ORDER BY total_orders DESC 
LIMIT 5;

customer_id,total_orders
C7381,9
C6444,8
C2231,8
C6672,8
C0362,8


In [0]:
SELECT 
    item.item_name, 
    COUNT(*) AS frequency 
FROM (
    SELECT EXPLODE(items_ordered) 
    AS item FROM alan_orders_bronze
)
GROUP BY item.item_name 
ORDER BY frequency DESC 
LIMIT 10;

item_name,frequency
Cell Garlic Bread,161
Magazine Pasta,152
Painting Coffee,151
Tough Garlic Bread,138
Section Coffee,132
Huge Nachos,129
Argue Cake,111
Have Tea,107
Purpose Brownie,104
Even Tea,103


In [0]:
SELECT 
    CEIL(AVG(SIZE(items_ordered)) 
    * 100) / 100 
    AS avg_items_per_order 
FROM alan_orders_bronze;

avg_items_per_order
1.65


In [0]:
SELECT
    restaurant_id,
    item.item_name,
    CEIL(SUM(item.price * item.quantity) * 100) / 100 AS item_revenue
FROM (
    SELECT
        restaurant_id,
        EXPLODE(items_ordered) AS item
    FROM alan_orders_bronze
)
GROUP BY
    restaurant_id,
    item.item_name
ORDER BY
    restaurant_id,
    item_revenue DESC;

restaurant_id,item_name,item_revenue
R001,Section Coffee,3641.8
R001,Type Ice Cream,2556.05
R001,Theory Spring Roll,1107.0
R001,Question Biryani,980.13
R001,Sort Brownie,964.17
R002,Skill Coffee,5637.5
R002,These Brownie,4677.76
R002,Allow Pasta,3629.42
R003,Cell Garlic Bread,6952.69
R003,Guess Pasta,6610.25


In [0]:
WITH customer_restaurant_counts AS (
    SELECT
        o.customer_id,
        c.name AS customer_name,
        r.name AS restaurant_name,
        COUNT(o.order_id) AS order_count
    FROM alan_orders_bronze o
    JOIN alan_customers_filtered c
        ON o.customer_id = c.customer_id
    JOIN alan_restaurants_filtered r
        ON o.restaurant_id = r.restaurant_id
    GROUP BY
        o.customer_id,
        c.name,
        r.name
),

ranked_restaurants AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY order_count DESC, restaurant_name ASC
        ) AS rn
    FROM customer_restaurant_counts
)

SELECT
    customer_name,
    restaurant_name,
    order_count
FROM ranked_restaurants
WHERE rn = 1
ORDER BY customer_name;

customer_name,restaurant_name,order_count
Aaron Arroyo,Food Haven Hub,1
Aaron Beard,Tasty Bites Hub,1
Aaron Beltran,Food Haven Grill,1
Aaron Bennett,Spice Villa Corner,1
Aaron Brown,Curry Leaf Corner,1
Aaron Carey,Pasta Palace Corner,1
Aaron Cooley,Pasta Palace Hub,1
Aaron Dalton,Spice Villa Corner,1
Aaron Diaz,Food Haven Grill,1
Aaron Dixon,Curry Leaf Express,1


In [0]:
WITH AreaMonthlyCounts AS (
    SELECT 
        MONTH(CAST(o.timestamp AS TIMESTAMP)) 
        AS order_month, 
        l.area AS area_name, 
        COUNT(o.order_id) AS total_orders,
        ROW_NUMBER() OVER 
        (PARTITION BY MONTH(CAST(o.timestamp 
        AS TIMESTAMP)) 
        ORDER BY COUNT(o.order_id) DESC) 
        as rank
    FROM alan_orders_bronze o
    JOIN alan_locations_filtered l 
    ON o.delivery_location_id = l.location_id
    GROUP BY MONTH(CAST(o.timestamp AS TIMESTAMP)), l.area
)
SELECT order_month, area_name, total_orders 
FROM AreaMonthlyCounts 
WHERE rank <= 10 
ORDER BY order_month, rank;

order_month,area_name,total_orders
1,Rodriguez Mill,87
1,Eric Hills,86
1,Hill Station,80
1,Smith Estates,79
1,Harper Street,76
1,Morgan Key,75
1,Copeland Parks,75
1,John Underpass,74
1,Jennifer Wall,73
1,Ashley Extensions,73


In [0]:
SELECT *
FROM alan_orders_bronze
WHERE LOWER(payment_method) = 'card'
  AND LOWER(order_status) = 'delivered';

agent_id,customer_id,delivery_location_id,items_ordered,order_id,order_status,payment_method,restaurant_id,timestamp,tip,total_amount
A086,C5605,L089,"List(List(I0491, Talk Pasta, 10.6, 2))",e293e41d-e335-4d6b-8504-34f96140e660,delivered,card,R006,2024-01-15T14:20:24,4.5,21.2
A352,C7525,L082,"List(List(I0403, Outside Juice, 23.61, 1), List(I0018, Finish Nachos, 6.76, 5), List(I0384, Mrs Cake, 9.28, 2))",c8bd7ee7-a451-447a-811b-fd69aff36062,delivered,card,R053,2024-01-04T07:36:42,0.3,75.97
A068,C1909,L083,"List(List(I0038, Near Soda, 15.97, 4), List(I0134, Structure Wrap, 16.48, 3))",e71e1b46-c026-4913-9b58-870fde58e72b,delivered,card,R047,2024-01-13T20:37:01,4.41,113.32
A196,C3246,L022,"List(List(I0185, Today Nachos, 22.46, 5))",ede3312f-d227-42a5-a1d0-d191999abbd1,delivered,card,R136,2024-01-02T20:57:26,1.4,112.3
A202,C8344,L100,"List(List(I0160, Almost Brownie, 13.71, 4), List(I0187, Travel Tea, 7.78, 2))",7e5fb9b0-c7e0-4080-89f3-6707df2a16a4,delivered,card,R074,2024-01-25T00:13:09,3.53,70.4
A390,C3024,L051,"List(List(I0254, Sell Ice Cream, 27.41, 2))",22accf1f-f43d-4051-9240-9a971c27bf85,delivered,card,R039,2024-01-20T00:43:15,1.31,54.82
A072,C2294,L082,"List(List(I0388, Nor Brownie, 9.17, 2))",bee913bc-f4fd-44b7-8f51-229c93ef4c5d,delivered,card,R018,2024-01-19T19:54:39,0.91,18.34
A307,C8811,L024,"List(List(I0112, Run Pizza, 25.36, 1), List(I0049, Soldier Ice Cream, 21.53, 2), List(I0441, Power Biryani, 0.0, 4))",5d838bcb-e367-4734-93fe-c9a2551e440f,delivered,card,R190,2024-01-05T14:06:33,2.46,68.42
A279,C1179,L038,"List(List(I0470, Radio Brownie, 25.75, 2))",938839a3-20ac-4b3b-a9b1-fb9b364bc210,delivered,card,R159,2024-01-02T16:18:53,4.28,51.5
A157,C2504,L062,"List(List(I0066, Reason Cake, 27.3, 1), List(I0158, Third Spring Roll, 18.26, 5))",3cd888ae-1614-4d75-bd5c-037edd3684f4,delivered,card,R110,2024-01-20T11:43:04,4.19,118.6


In [0]:
SELECT 
    r.cuisines, 
    CEIL(AVG(o.total_amount) * 100) 
    / 100 AS avg_order_value
FROM alan_orders_bronze o
JOIN alan_restaurants_filtered r 
ON o.restaurant_id = r.restaurant_id
GROUP BY r.cuisines
ORDER BY avg_order_value DESC;

cuisines,avg_order_value
"Thai,Chinese",160.11
"Thai,Italian,Chinese",156.69
"Mexican,American",146.19
"Mexican,Indian,Thai",144.91
"Mexican,American,Thai",143.95
"American,Mexican",123.97
"Italian,Indian,Mexican",121.06
"American,Chinese",118.96
"American,Italian",110.58
"Chinese,Mexican",109.4


In [0]:
SELECT 
    a.name AS agent_name, 
    COUNT(o.order_id) 
    AS high_value_orders
FROM alan_orders_bronze o
JOIN alan_delivery_agents_filtered a 
ON o.agent_id = a.agent_id
WHERE o.total_amount > 50
GROUP BY a.name
ORDER BY high_value_orders DESC;

agent_name,high_value_orders
Christopher Chase,213
Christopher Williams,171
Justin Lam,118
Charlene Morgan,112
Joan Harrison,110
Carrie Maxwell,108
John King,108
Justin Smith,106
James Osborn,104
Teresa Ryan,102


In [0]:
SELECT 
    l.area, 
    CEIL(SUM(o.total_amount) * 100) 
    / 100 
    AS total_revenue
FROM alan_orders_bronze o
JOIN alan_restaurants_filtered r 
ON o.restaurant_id = r.restaurant_id
JOIN alan_locations_filtered l 
ON r.location_id = l.location_id
WHERE r.cuisines = 'Italian'
GROUP BY l.area
ORDER BY total_revenue DESC;

area,total_revenue
Henderson Creek,11356.8
Burns Burg,4822.74
Williams Way,3168.24


In [0]:
SELECT 
    r.cuisines, 
    CEIL(AVG(try_divide(o.tip, o.total_amount)) 
    * 100) 
    / 100 
    AS tip_to_total_ratio
FROM alan_orders_bronze o
JOIN alan_restaurants_filtered r 
ON o.restaurant_id = r.restaurant_id
GROUP BY r.cuisines
ORDER BY tip_to_total_ratio DESC;

cuisines,tip_to_total_ratio
"Indian,Mexican,American",0.22
"Thai,American,Indian",0.17
"Chinese,Thai",0.16
"Chinese,American",0.11
"Italian,Thai,Chinese",0.1
"Chinese,Italian,Thai",0.1
"Chinese,Indian,American",0.09
"American,Italian,Indian",0.08
"Indian,American",0.08
"Indian,American,Mexican",0.08


In [0]:
SELECT 
    i.item_name, 
    r.name AS restaurant_name, 
    COUNT(*) AS order_frequency
FROM (
    SELECT restaurant_id, EXPLODE(items_ordered) 
    AS item FROM alan_orders_bronze
) exploded_orders
JOIN alan_menu_items_filtered i 
ON exploded_orders.item.item_id = i.item_id
JOIN alan_restaurants_filtered r 
ON exploded_orders.restaurant_id = r.restaurant_id
GROUP BY i.item_name, r.name
ORDER BY order_frequency DESC
LIMIT 10;

item_name,restaurant_name,order_frequency
Argue Cake,Curry Leaf Kitchen,111
Even Tea,Food Haven Kitchen,103
Rich Biryani,Curry Leaf Corner,101
Sell Ice Cream,Food Haven Grill,101
Finish Garlic Bread,Spice Villa Hub,100
Market Spring Roll,Pasta Palace Express,99
Student Tea,Pasta Palace Corner,99
Thought Pizza,Tasty Bites Grill,98
Bring Cake,Curry Leaf Corner,98
Street Pizza,Curry Leaf Corner,98


In [0]:
SELECT 
    a.name AS agent_name, 
    COUNT(DISTINCT r.cuisines) 
    AS distinct_cuisines_delivered
FROM alan_orders_bronze o
JOIN alan_restaurants_filtered r 
ON o.restaurant_id = r.restaurant_id
JOIN alan_delivery_agents_filtered a 
ON o.agent_id = a.agent_id
GROUP BY a.name
ORDER BY distinct_cuisines_delivered DESC;

agent_name,distinct_cuisines_delivered
Christopher Chase,43
Justin Smith,40
Justin Lam,39
Christopher Williams,39
James Osborn,38
Carrie Maxwell,37
Teresa Ryan,37
Thomas Lowe,36
Jamie Jones,36
David King,35


In [0]:
WITH CustomerSpend AS (
    SELECT 
        YEAR(CAST(c.signup_date AS DATE)) 
        AS signup_year, 
        c.customer_id, 
        SUM(o.total_amount) 
        AS total_spend
    FROM alan_orders_bronze o
    JOIN alan_customers_filtered c 
    ON o.customer_id = c.customer_id
    GROUP BY YEAR(CAST(c.signup_date 
    AS DATE)), c.customer_id
)
SELECT 
    signup_year, 
    CEIL(AVG(total_spend) 
    * 100) / 100 
    AS avg_customer_ltv
FROM CustomerSpend
GROUP BY signup_year
ORDER BY signup_year;

signup_year,avg_customer_ltv
1954,22.64
1955,172.65
1956,139.91
1957,155.42
1958,135.54
1959,141.09
1960,173.02
1961,222.25
1962,192.06
1963,137.27


In [0]:
SELECT 
    r.name AS restaurant_name, 
    l.area AS location_area, 
    COUNT(o.order_id) AS order_frequency
FROM alan_orders_bronze o
JOIN alan_restaurants_filtered r 
ON o.restaurant_id = r.restaurant_id
JOIN alan_locations_filtered l 
ON o.delivery_location_id = l.location_id
GROUP BY r.name, l.area
ORDER BY order_frequency DESC;

restaurant_name,location_area,order_frequency
Spice Villa Grill,Joshua Drive,21
Curry Leaf Corner,Carpenter Views,21
Spice Villa Hub,Rodriguez Mill,19
Curry Leaf Corner,John Underpass,18
Curry Leaf Corner,Rodriguez Mill,18
Spice Villa Grill,Reed Glens,18
Spice Villa Grill,Gary Green,18
Spice Villa Grill,Torres Mountain,18
Spice Villa Grill,Martinez Route,17
Curry Leaf Corner,Flores Vista,17


In [0]:
SELECT 
    l.area AS area_name, 
    COUNT(o.order_id) 
    AS cancellation_count
FROM alan_orders_bronze o
JOIN alan_locations_filtered l 
ON o.delivery_location_id = l.location_id
WHERE LOWER(o.order_status) = 'cancelled'
GROUP BY l.area
ORDER BY cancellation_count DESC;

area_name,cancellation_count
Evans Stream,58
Elizabeth Locks,57
Flores Vista,55
Ashley Extensions,53
Hill Station,52
Padilla Mall,52
White Tunnel,51
Joshua Drive,51
Curtis Oval,51
Rodriguez Mill,51


In [0]:
SELECT
    c.name AS customer_name,
    COUNT(DISTINCT o.restaurant_id) AS distinct_restaurants
FROM alan_orders_bronze o
JOIN alan_customers_filtered c
    ON o.customer_id = c.customer_id
GROUP BY
    c.customer_id,
    c.name
ORDER BY
    distinct_restaurants DESC,
    customer_name ASC
LIMIT 10;

customer_name,distinct_restaurants
Thomas Jennings,9
Alejandro Rice,8
Eric Carroll,8
Mr. Eddie Wells MD,8
Peter Conway,8
Carlos Tyler,7
Christopher Cunningham,7
Elizabeth Campos,7
Gabriela Phillips,7
Heather Watson DDS,7


In [0]:
SELECT
    r.name AS restaurant_name,
    c.name AS customer_name,
    COUNT(o.order_id) AS order_count
FROM alan_orders_bronze o
JOIN alan_restaurants_filtered r
    ON o.restaurant_id = r.restaurant_id
JOIN alan_customers_filtered c
    ON o.customer_id = c.customer_id
GROUP BY
    r.restaurant_id,
    r.name,
    c.customer_id,
    c.name
HAVING COUNT(o.order_id) > 1
ORDER BY
    order_count DESC,
    restaurant_name ASC,
    customer_name ASC;

restaurant_name,customer_name,order_count
Curry Leaf Corner,Jose Alexander,4
Curry Leaf Corner,April Cummings,2
Curry Leaf Corner,Ashley Carey,2
Curry Leaf Corner,Ashley Thompson,2
Curry Leaf Corner,Brent Nichols,2
Curry Leaf Corner,Dustin Hansen,2
Curry Leaf Corner,Joan Smith,2
Curry Leaf Corner,Karen Randall,2
Curry Leaf Corner,Kelli Luna,2
Curry Leaf Corner,Kevin Wheeler,2


In [0]:
SELECT 
    r.cuisines, 
    o.payment_method, 
    COUNT(o.order_id) 
    AS usage_count
FROM alan_orders_bronze o
JOIN alan_restaurants_filtered r 
ON o.restaurant_id = r.restaurant_id
WHERE o.total_amount > 40
GROUP BY r.cuisines, o.payment_method
ORDER BY r.cuisines, usage_count DESC;

cuisines,payment_method,usage_count
American,card,125
American,UPI,122
American,cash,118
American,wallet,117
"American,Chinese",card,46
"American,Chinese",UPI,46
"American,Chinese",wallet,34
"American,Chinese",cash,28
"American,Chinese,Italian",wallet,22
"American,Chinese,Italian",UPI,21
